# Data Formatting

The scripts here convert datasets into the formats used for the dataanalysis

## Events Manual Labels

Convert events encodings from CSV to NC

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

events_coding_files = ["waddendrifters2023_grounding_events_manual_coding","waddendrifters2023_grounding_events_manual_coding_validation"]

for ec_file in events_coding_files:
  df = pd.read_csv("data/in/%s.csv"%ec_file, engine="c",header=5)
  indices = np.array([[int(n) for n in ts_name.split('_')] for ts_name in df['ts_name']])
  df['irecord'] = indices[:,0]
  df['its'] = indices[:,1]
  df = df.drop(columns=['ts_name', 'comment'])
  ds = xr.Dataset.from_dataframe(df)
  sections = np.unique([[int(ir),int(its)] for ir in ds.irecord for its in ds.its],axis=0)
  ds.attrs = {"occurring_irecord" : sections[:,0], "occurring_its" : sections[:,1]}
  ds.to_netcdf('data/out/%s.nc'%ec_file)
  print("written data/out/%s.nc"%ec_file)

ds

written data/out/waddendrifters2023_grounding_events_manual_coding.nc
written data/out/waddendrifters2023_grounding_events_manual_coding_validation.nc


<xarray.Dataset> Size: 4kB
Dimensions:     (index: 92)
Coordinates:
  * index       (index) int64 736B 0 1 2 3 4 5 6 7 8 ... 84 85 86 87 88 89 90 91
Data variables:
    ind_start   (index) int64 736B 186 190 1157 1210 ... 1184 1213 1226 1227
    tag         (index) object 736B 'O' 'O' 'D' 'W' 'D' ... 'W' 'D' 'W' 'D' 'D'
    edit notes  (index) object 736B nan nan nan nan nan ... nan nan nan nan nan
    irecord     (index) int64 736B 1 1 1 1 1 1 1 1 1 ... 24 24 26 26 26 26 26 26
    its         (index) int64 736B 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
Attributes:
    occurring_irecord:  [ 1  5  9 13 18 20 21 23 24 26]
    occurring_its:      [0 0 0 0 0 0 0 0 0 0]

## Optional: Format data from https://waterinfo.rws.nl/publiek/waterhoogte/ (CSV >> NC)

In [12]:
file_name_input_csv = "rws_waterlevels_14Nov_1Dec.csv"
file_name_output_nc = file_name_input_csv[:-3] + "nc"

import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime

def epoch_timestamp_from_timestringYMDHMSZ(timestringYMDHMSZ):
  sformat = '%d-%m-%YT%H:%M:%S %Z'
  t1 = datetime.strptime(timestringYMDHMSZ, sformat)
  t0 = datetime.strptime('01-01-1970T00:00:00 %s'%timestringYMDHMSZ[20:], sformat)
  return (t1-t0).total_seconds()

table = pd.read_csv("data/in/"+file_name_input_csv, delimiter=';')

n_rows = len(table)
table_timestamps = np.array([epoch_timestamp_from_timestringYMDHMSZ("%sT%s CET" % (table.WAARNEMINGDATUM[i], table.WAARNEMINGTIJD[i])) for i in range(n_rows)])
times = np.unique(table_timestamps)
n_times = len(times)
location_codes = np.unique(table.LOCATIE_CODE)
n_locs = len(location_codes)
locs = np.zeros((n_locs,2)) #(lon,lat)
for i in range(n_locs):
  iloc = np.argwhere(table.LOCATIE_CODE==location_codes[i])[0]
  iloc = iloc if isinstance(iloc, int) else iloc[0]
  locs[i,0], locs[i,1] = table.LON[iloc], table.LAT[iloc]
waterlevels = np.zeros((n_times,n_locs))
for it in range(n_times):
  for il in range(n_locs):
    ind = np.argwhere((table_timestamps==times[it])&(table.LOCATIE_CODE==location_codes[il]))[0]
    ind = ind if isinstance(ind, int) else ind[0]
    waterlevels[it,il] = table.NUMERIEKEWAARDE.values[ind]/100.

dat = xr.Dataset(
  data_vars = {
    "timestamp": (["itime"],times),
    "lon": (["ipos"],locs[:,0]),
    "lat": (["ipos"],locs[:,1]),
    "waterlevel":  (["itime","ipos"],waterlevels),
  },
  coords={
    "itime": np.array(range(n_times)),
    "ipos": np.array(range(n_locs))
  })
dat.attrs = {"units" : "s, m (NAP), deg"}
dat.to_netcdf('data/out/'+file_name_output_nc)

dat

<xarray.Dataset> Size: 125kB
Dimensions:     (itime: 2593, ipos: 4)
Coordinates:
  * itime       (itime) int64 21kB 0 1 2 3 4 5 ... 2587 2588 2589 2590 2591 2592
  * ipos        (ipos) int64 32B 0 1 2 3
Data variables:
    timestamp   (itime) float64 21kB 1.7e+09 1.7e+09 ... 1.701e+09 1.701e+09
    lon         (ipos) float64 32B 5.759 4.745 5.409 5.091
    lat         (ipos) float64 32B 53.43 52.96 53.18 53.3
    waterlevel  (itime, ipos) float64 83kB 1.37 0.27 1.37 0.71 ... 0.45 1.2 0.95
Attributes:
    units:    s, m (NAP), deg